<!-- NB_DOC_INTRO_V1 -->
# Preprocessing — snippets utilitaires bonus

> 📚 **Doc thematique** : [docs/01_STRUCTURES.md](docs/01_STRUCTURES.md) (Structures Python / DataFrame / Preprocessing)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Recueil de **snippets courts et reutilisables** pour le preprocessing data : conversions robustes, manipulation de variables Python, raccourcis pandas. Annexe a [Structures_Preprocessing.ipynb](./Structures_Preprocessing.ipynb).

## Plan

1. Conversion FR float (virgule decimale)
2. Detection auto colonnes par dtype
3. Down-casting memoire
4. np.select multi-conditions
5. df.pipe chaining + tqdm.progress_apply
6. Recherche dans listes/dicts
7. Style pandas (formatage notebook)
8. References


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Conversion FR float (virgule decimale)

In [ ]:
def to_float_fr(s):
    """Gere les '1,5' (virgule decimale FR) et '1 000,5' (espace milliers)."""
    if pd.isna(s) or s == "":
        return np.nan
    s = str(s).replace(" ", "").replace(" ", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan

# Demo
samples = ["1,5", "1 000,5", "1.5", "abc", None, "2,0"]
for s in samples:
    print(f"  {s!r:15s} → {to_float_fr(s)}")

## 2. Detection auto colonnes par dtype

In [ ]:
df = pd.DataFrame({
    "age": [25, 30, 35],
    "salary": [30000.5, 45000.0, 50000.5],
    "name": ["Alice", "Bob", "Charlie"],
    "joined": pd.to_datetime(["2020-01-01", "2021-06-15", "2022-03-20"]),
    "active": [True, False, True],
    "category": pd.Categorical(["A", "B", "A"]),
})

num_cols = df.select_dtypes(include=np.number).columns.tolist()
str_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
cat_cols = df.select_dtypes(include="category").columns.tolist()
date_cols = df.select_dtypes(include="datetime").columns.tolist()
bool_cols = df.select_dtypes(include="bool").columns.tolist()

print(f"Numeriques  : {num_cols}")
print(f"Strings     : {str_cols}")
print(f"Categories  : {cat_cols}")
print(f"Dates       : {date_cols}")
print(f"Booleens    : {bool_cols}")

## 3. Down-casting memoire

In [ ]:
# Reduit l'empreinte memoire en down-castant les types
def downcast_numeric(df):
    out = df.copy()
    for col in out.select_dtypes(include="integer").columns:
        out[col] = pd.to_numeric(out[col], downcast="integer")
    for col in out.select_dtypes(include="float").columns:
        out[col] = pd.to_numeric(out[col], downcast="float")
    return out

# Conversion strings → category pour col a faible cardinalite (gain massif memoire)
def cat_low_cardinality(df, max_unique_ratio=0.5):
    out = df.copy()
    for col in out.select_dtypes(include=["object", "string"]).columns:
        if out[col].nunique() / len(out) < max_unique_ratio:
            out[col] = out[col].astype("category")
    return out

# Demo
big = pd.DataFrame({
    "id": np.arange(100_000),
    "score": np.random.randn(100_000),
    "country": np.random.choice(["FR", "DE", "ES", "IT"], 100_000),
})
print(f"Memoire avant : {big.memory_usage(deep=True).sum():,} bytes")
big_opt = downcast_numeric(cat_low_cardinality(big))
print(f"Memoire apres : {big_opt.memory_usage(deep=True).sum():,} bytes")
print(f"Gain : {1 - big_opt.memory_usage(deep=True).sum() / big.memory_usage(deep=True).sum():.1%}")

## 4. np.select — multi-conditions lisible

In [ ]:
# Alternative aux np.where imbriques (vite illisibles)
df = pd.DataFrame({"age": [15, 25, 50, 80, 35]})

conditions = [
    df["age"] < 18,
    df["age"] < 60,
    df["age"] < 100,
]
choices = ["mineur", "actif", "senior"]
df["categorie"] = np.select(conditions, choices, default="inconnu")
print(df)

## 5. df.pipe chaining + tqdm.progress_apply

In [ ]:
# pipe pour chainer plusieurs traitements
def clean_text(s):
    return s.str.strip().str.lower()

def remove_outliers(df, col, k=3):
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    return df[(df[col] >= q1 - k * iqr) & (df[col] <= q3 + k * iqr)]

# Chainage propre
processed = (df
    .assign(age_doubled=lambda x: x["age"] * 2)
    .pipe(remove_outliers, col="age", k=3)
    .pipe(lambda x: x.assign(adult=x["age"] >= 18))
)
print(processed)

# tqdm.progress_apply pour barre de progression sur apply
# from tqdm import tqdm; tqdm.pandas()
# df["new_col"] = df["text"].progress_apply(expensive_func)
print("\ntqdm.progress_apply : decommenter pour usage reel")

## 6. Recherche dans listes / dicts

In [ ]:
# Dans une liste de dicts
data = [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}, {"id": 3, "name": "Charlie"}]

# Find first match
target = next((x for x in data if x["id"] == 2), None)
print(f"First match : {target}")

# All matches
all_matches = [x for x in data if x["id"] >= 2]
print(f"All matches : {all_matches}")

# Dans une liste : indices d'un element
ma_liste = [1, 2, 3, 2, 4, 2]
val = 2
indices = [i for i, x in enumerate(ma_liste) if x == val]
print(f"\nIndices de {val} dans la liste : {indices}")

## 7. Style pandas — formatage pour notebooks

In [ ]:
# df.style permet de formater l'affichage (couleurs, % , etc.)
demo = pd.DataFrame({
    "metric": ["accuracy", "precision", "recall"],
    "score": [0.876, 0.923, 0.654],
    "change": [0.012, -0.005, 0.023],
})

styled = (demo.style
    .format({"score": "{:.1%}", "change": "{:+.2%}"})
    .background_gradient(cmap="RdYlGn", subset=["score"])
    .map(lambda v: "color: green" if v > 0 else "color: red", subset=["change"])
)
# En notebook : displayer simplement styled
# Pour exporter : styled.to_html("styled.html")
print(f"Pandas Styler cree avec {len(demo)} rows + formatting")
print("En notebook reel : displayer la variable 'styled' directement.")

## References

### Documentation
- pandas Styler : https://pandas.pydata.org/docs/user_guide/style.html
- tqdm pandas : https://tqdm.github.io/docs/notebook/

### Voir aussi
- [Structures_DataFrame.ipynb](Structures_DataFrame.ipynb)
- [Structures_Preprocessing.ipynb](Structures_Preprocessing.ipynb)
- [Structure_Pyhton.ipynb](Structure_Pyhton.ipynb)
